In [0]:
import uuid
from datetime import datetime, timezone, date
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

ESQUEMA_LANDING = StructType([
    StructField("_source_id", StringType(), True),
    StructField("_body", StringType(), True),
])

class BronzeWriter:
    def __init__(self, spark, catalog, schema):
        self.spark = spark
        self.catalog = catalog
        self.schema = schema

    def write_from_landing(self, file_path, collection, load_type):
        if file_path is None:
            return 0

        landing_df = self.spark.read.schema(ESQUEMA_LANDING).json(file_path)
        qtd = landing_df.count()  # conta a partir da landing, independente do tmp_path

        tmp_path = file_path.replace(".json", "_body_tmp")
        landing_df.select(F.col("_body").alias("value")).write.mode("overwrite").text(tmp_path)

        estruturado = (self.spark.read
                         .option("primitivesAsString", "true")
                         .option("columnNameOfCorruptRecord", "_rescued_data")
                         .json(tmp_path))

        if "_rescued_data" not in estruturado.columns:
            estruturado = estruturado.withColumn("_rescued_data", F.lit(None).cast("string"))

        df = (estruturado
                .withColumnRenamed("_id", "_source_id")
                .withColumn("_ingestion_id", F.lit(str(uuid.uuid4())))
                .withColumn("_ingestion_timestamp", F.lit(datetime.now(timezone.utc)))
                .withColumn("_source_path", F.lit("mongodb_atlas"))
                .withColumn("_load_type", F.lit(load_type))
                .withColumn("_ingestion_date", F.lit(date.today().isoformat())))

        nome = f"{self.catalog}.{self.schema}.{collection}"
        if not self.spark.catalog.tableExists(nome):
            (df.write.format("delta").mode("append")
               .option("mergeSchema", "true")
               .partitionBy("_ingestion_date")
               .saveAsTable(nome))
        else:
            (DeltaTable.forName(self.spark, nome).alias("t")
                .merge(df.alias("s"), "t._source_id = s._source_id AND t._ingestion_date = s._ingestion_date")
                .whenNotMatchedInsertAll()
                .execute())

        dbutils.fs.rm(tmp_path, recurse=True)  # agora seguro: write já consumiu o tmp_path acima

        return qtd